In [1]:
%pip install jupysql duckdb-engine matplotlib


You should consider upgrading via the '/Users/saikrishnakadupudi/.pyenv/versions/3.10.1/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install grpcio

You should consider upgrading via the '/Users/saikrishnakadupudi/.pyenv/versions/3.10.1/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [3]:
%reload_ext sql

In [4]:
%sql duckdb:///barista.coffee.db

Connecting to 'duckdb:///barista.coffee.db'

In [5]:
%%sql
create or replace Table orders as
from read_csv_auto('barista_coffee_sales_data_for_eda.csv',ignore_errors=1);

Running query in 'duckdb:///barista.coffee.db'

Count


In [6]:
%%sql
select count(*)
from orders;

Running query in 'duckdb:///barista.coffee.db'

count_star()
100000


In [7]:
%%sql
select *
from orders 
limit 5;

Running query in 'duckdb:///barista.coffee.db'

order_date,order_time,product_category,qty,unit_price,store_id,store_location,customer_id,customer_age,customer_gender,loyalty_member,is_repeat_customer,customer_discovery_source,payment_method,discount_applied,total_amount,promo_applied,promo_type
2023-04-29,22:47:20,Pizza,3,7.08,STORE_7,Airport,CUST_1,36,Female,False,False,Walk-in,Cash,0.23,29.27,False,None
2023-02-26,09:44:20,Coffee,4,4.97,STORE_10,Suburbs,CUST_2,39,Male,True,True,Walk-in,Wallet,0.46,36.52,True,None
2021-08-08,20:50:40,Coffee,1,3.17,STORE_4,Airport,CUST_3,19,Female,True,False,Social Media,Card,0.24,13.17,False,Flat discount
2022-01-29,16:31:01,Pizza,3,6.42,STORE_4,Suburbs,CUST_4,25,Female,True,False,Walk-in,UPI,0.35,17.88,False,Flat discount
2022-04-12,07:18:01,Snacks,3,4.51,STORE_8,Uptown,CUST_5,53,Male,False,True,Friend,Cash,0.32,45.61,True,Flat discount


# Lets get sub urbs , category wise sales for whole months


In [8]:
%%sql 
create or replace View Suburbs as
select customer_id,order_date,product_category,qty,store_location
from orders;

Running query in 'duckdb:///barista.coffee.db'

Count


In [9]:
%%sql
with cte as
(SELECT 
    product_category, 
    SUM(case when store_location = 'Suburbs' then qty else 0 end) as 'Suburbs_Total_Quanity',
    SUM(case when store_location = 'Airport' then qty else 0 end) as 'Airport_Total_Quanity',
    Row_number() over (partition by Month(order_date)) as rank,
    MONTH(order_date) AS Month_of_2023
FROM 
    Suburbs
WHERE YEAR(order_date) = 2023
GROUP BY 
    MONTH(order_date), product_category
ORDER BY 
    5, 1)
select * exclude rank from cte
where rank <=3;


Running query in 'duckdb:///barista.coffee.db'

product_category,Suburbs_Total_Quanity,Airport_Total_Quanity,Month_of_2023
Cold Drinks,106,178,1
Merchandise,156,156,1
Tea,152,114,1
Coffee,128,152,2
Cold Drinks,131,125,2
Tea,123,117,2
Cold Drinks,142,120,3
Merchandise,167,147,3
Tea,141,140,3
Cold Drinks,145,102,4


conn.rollback()

Lets get some -----Customer Behavior & Demographics ------Details

In [10]:
%%sql
with cte as
(select total_amount,
case 
    when customer_age < 25 then 'Less than 25'
    when customer_age <= 35 and customer_age >= 25 then '25-35'
    when customer_age <=50 and customer_age >= 36 then '36-50'
    else '>50' end as 'Age_group'
from orders)
select Age_group,Round(AVG(total_amount),2) as 'Amount'
from cte
group by Age_group;

Running query in 'duckdb:///barista.coffee.db'

Age_group,Amount
Less than 25,27.44
25-35,27.54
36-50,27.36
>50,27.52


Lets take total amount spent by gender distribution as well.


In [11]:
%%sql 
select customer_gender, round(SUM(total_amount),2) as 'SUM of Amount spent', round(AVG(total_amount),2) as 'Avg of amount spent'
from orders
group by customer_gender;

Running query in 'duckdb:///barista.coffee.db'

customer_gender,SUM of Amount spent,Avg of amount spent
Other,926870.9,27.58
Female,913137.42,27.42
Male,906259.77,27.38


Do loyalty members spend more per transaction than non-members?


In [12]:
%%sql
select loyalty_member, SUM(total_amount) as 'sum of amount spent', avg(total_amount) as 'avg of amount spent'
from orders
group by 1;

Running query in 'duckdb:///barista.coffee.db'

loyalty_member,sum of amount spent,avg of amount spent
False,1379757.799999999,27.520849705794337
True,1366510.290000005,27.404197132257192


Which store_location (Airport, Suburbs, Uptown) has the highest total sales?

In [13]:
%%sql
select store_location,avg(total_amount),round(sum(total_amount),2)
from orders
group by store_location
order by 3 desc;

Running query in 'duckdb:///barista.coffee.db'

store_location,avg(total_amount),"round(sum(total_amount), 2)"
Downtown,27.514813325903557,691282.17
Suburbs,27.468337443321886,690608.94
Uptown,27.319427496196607,682384.66
Airport,27.5485668120858,681992.32


In [14]:
#Which store_id generated the most revenue in 2023

In [15]:
%%sql
select store_id as 'Top Revenue store IDs'
from orders
group by store_id
order by sum(total_amount) desc
limit 5;

Running query in 'duckdb:///barista.coffee.db'

Top Revenue store IDs
STORE_2
STORE_3
STORE_8
STORE_4
STORE_9


In [16]:
%%sql
select payment_method,count(*) * 100/ (select count(*) from orders)
from orders
group by payment_method;


Running query in 'duckdb:///barista.coffee.db'

payment_method,((count_star() * 100) / (SELECT count_star() FROM orders))
UPI,25.291
Wallet,24.932
Card,24.895
Cash,24.882


Is there a difference in average order value between weekdays and weekends?

In [22]:
%%sql
with cte as 
(select 
case when weekday(order_date) > 4 then False else True end as is_weekday,avg(total_amount)
from orders
group by is_weekday)
select * from cte;

Running query in 'duckdb:///barista.coffee.db'

is_weekday,avg(total_amount)
True,27.47358540350893
False,27.43565669565235


At what hours are most orders placed

In [23]:
%%sql
select * from orders limit 1;

Running query in 'duckdb:///barista.coffee.db'

order_date,order_time,product_category,qty,unit_price,store_id,store_location,customer_id,customer_age,customer_gender,loyalty_member,is_repeat_customer,customer_discovery_source,payment_method,discount_applied,total_amount,promo_applied,promo_type
2023-04-29,22:47:20,Pizza,3,7.08,STORE_7,Airport,CUST_1,36,Female,False,False,Walk-in,Cash,0.23,29.27,False,None


In [ ]:
%%sql
select Hour(order_time) as "hour_of_day", sum(qty) as "No_of_products"
from orders
group by hour_of_day
order by 2;

Running query in 'duckdb:///barista.coffee.db'

hour_of_day,No_of_products
6,10376
4,10403
13,10407
17,10411
9,10439
11,10444
22,10486
16,10488
3,10520
14,10538
